In [4]:
from mpqp.all import *

In [20]:
from mpqp.core.instruction.measurement.pauli_string import I, PauliStringMonomial


def pauli_monomial_eigenvalues(monom: PauliStringMonomial):
    result = np.array([1])
    eigen_I = np.array([1, 1])
    eigen_XYZ = np.array([1, -1])
    for atom in monom.atoms:
        if atom == I:
            result = np.kron(result, eigen_I)
        else:
            result = np.kron(result, eigen_XYZ)
    return result

In [66]:
def find_every_rotations(groups: list[PauliStringMonomial]) -> list[Gate]:
    result = []
    for i, g1 in enumerate(groups):
        for j, g2 in enumerate(groups):
            if i <= j:
                continue
            for k in range(len(g1.atoms)):
                if (g1.atoms[k] == X) or (g2.atoms[k] == X):
                    result.append(Ry(-np.pi / 2, k))
                elif (g1.atoms[k] == Y) or (g2.atoms[k] == Y):
                    result.append(Rx(-np.pi / 2, k))
    return result

In [73]:
from mpqp.tools.pauli_grouping import CommutingTypes, pauli_grouping_greedy


def whole_process(monoms: list[PauliStringMonomial], circuit: QCircuit):
    groups = pauli_grouping_greedy(monoms, CommutingTypes.QUBITWISE)

    circuit.add(find_every_rotations(groups[0]))
    result = run(circuit, IBMDevice.AER_SIMULATOR)
    assert isinstance(result, Result)
    r1 = np.dot(pauli_monomial_eigenvalues(groups[0][0]), result.probabilities)
    r2 = np.dot(result.probabilities, pauli_monomial_eigenvalues(groups[0][1]))

    return [r1, r2]

In [74]:
def verify_results(monoms, circuit, previous_results):
    groups = pauli_grouping_greedy(monoms, CommutingTypes.QUBITWISE)
    res = []
    for group in groups:
        for monom in group:
            from copy import deepcopy

            cirq = deepcopy(circuit)
            cirq.add(ExpectationMeasure(Observable(monom)))
            res.append(run(cirq, IBMDevice.AER_SIMULATOR).expectation_values)
    return all(
        round(res[i], 10) == round(previous_results[i], 10) for i in range(len(res))
    )

In [75]:
from copy import deepcopy
from mpqp.core.instruction.measurement.pauli_string import I, X, Y, Z


circuit = QCircuit(3)
circuit.add(Rx(np.pi / 3, 0))
circuit.add(Rz(1 / 100, 0))
circuit.add(Ry(np.pi / 5, 1))
circuit.add(Rz(np.pi / 10, 2))
circuit.add(Rx(1 / 25, 2))
circuit.add(Rz(np.pi / 20, 1))
string = X @ Y @ I + X @ I @ Z

res = whole_process(string.monomials, deepcopy(circuit))
print(verify_results(string.monomials, circuit, res))

True


In [39]:
print(np.kron(np.kron(np.array([1, -1]), np.array([1, -1])), np.array([1, 1])))
print(np.kron(np.kron(np.array([1, -1]), np.array([1, 1])), np.array([1, -1])))

[ 1  1 -1 -1 -1 -1  1  1]
[ 1 -1  1 -1 -1  1 -1  1]
